In [4]:

# 1. Import required libraries

import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_val_score,
    GridSearchCV
)
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score


# 2. Load Breast Cancer Dataset

data = load_breast_cancer()

X = data.data
y = data.target

print("Dataset loaded successfully")
print("Total samples:", X.shape[0])
print("Total features:", X.shape[1])


# 3. Split into Training, Validation and Testing sets

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("\nDataset Splitting:")
print("Training samples:", len(X_train))
print("Validation samples:", len(X_val))
print("Testing samples:", len(X_test))


# 4. Train Logistic Regression (Baseline Model)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

baseline_model = LogisticRegression(max_iter=5000)

baseline_model.fit(X_train_scaled, y_train)

baseline_pred = baseline_model.predict(X_val_scaled)

baseline_accuracy = accuracy_score(
    y_val, baseline_pred
)

print("\nBaseline Logistic Regression")
print("Validation Accuracy:", round(baseline_accuracy, 4))


# 5. Perform 5-Fold Cross Validation

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

cv_scores = cross_val_score(
    baseline_model,
    X_train_scaled,
    y_train,
    cv=cv,
    scoring="accuracy"
)

print("\n5-Fold Cross Validation")
print("Fold Scores:", np.round(cv_scores, 4))
print("Average CV Accuracy:", round(cv_scores.mean(), 4))


# 6. Create Pipeline with StandardScaler and Logistic Regression

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=5000))
])

pipeline.fit(X_train, y_train)

pipeline_pred = pipeline.predict(X_val)

pipeline_accuracy = accuracy_score(
    y_val, pipeline_pred
)

print("\nPipeline Model")
print("Validation Accuracy:", round(pipeline_accuracy, 4))


# 7. Hyperparameter Tuning using GridSearchCV

param_grid = {
    "model__C": [0.01, 0.1, 1, 10, 100]
}

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=cv,
    scoring="accuracy",
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print("\nGridSearchCV Results")
print("Best Parameters:", grid_search.best_params_)
print("Best CV Accuracy:", round(grid_search.best_score_, 4))


# 8. Evaluate Tuned Model on Validation Data

best_model = grid_search.best_estimator_

tuned_pred = best_model.predict(X_val)

tuned_accuracy = accuracy_score(
    y_val, tuned_pred
)

print("\nTuned Model")
print("Validation Accuracy:", round(tuned_accuracy, 4))


# 9. Compare Model Results

print("\n----- MODEL COMPARISON -----")

print("Baseline Accuracy:",
      round(baseline_accuracy, 4))

print("Pipeline Accuracy:",
      round(pipeline_accuracy, 4))

print("Tuned Model Accuracy:",
      round(tuned_accuracy, 4))


# 10. Final Testing

test_pred = best_model.predict(X_test)

test_accuracy = accuracy_score(
    y_test, test_pred
)

print("\n----- FINAL TEST RESULT -----")
print("Test Accuracy:", round(test_accuracy, 4))

print("\nClassification completed successfully!")


Dataset loaded successfully
Total samples: 569
Total features: 30

Dataset Splitting:
Training samples: 398
Validation samples: 85
Testing samples: 86

Baseline Logistic Regression
Validation Accuracy: 0.9882

5-Fold Cross Validation
Fold Scores: [1.     0.975  0.9875 1.     0.9494]
Average CV Accuracy: 0.9824

Pipeline Model
Validation Accuracy: 0.9882

GridSearchCV Results
Best Parameters: {'model__C': 0.1}
Best CV Accuracy: 0.9849

Tuned Model
Validation Accuracy: 0.9882

----- MODEL COMPARISON -----
Baseline Accuracy: 0.9882
Pipeline Accuracy: 0.9882
Tuned Model Accuracy: 0.9882

----- FINAL TEST RESULT -----
Test Accuracy: 0.9767

Classification completed successfully!
